<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Diagnoses disagreements between final marks and finish/status fields in the student-course table.

**Notebook Shape:** 3 cells (2 code, 1 markdown).

**Inputs / Data Sources:**
- `df = pd.read_parquet(DATA_PATH)`
- `df = pd.read_csv(DATA_PATH, dtype="string")`

**Outputs / Side Effects:**
- `No explicit persisted output detected; side effects are limited to notebook display state unless cells are edited.`

**Logic Flow:**
1. Load raw or cleaned student-course data.
2. Compare mark values against finish/status indicators.
3. Inspect disagreement examples.

**Maintainability Notes:** Findings should feed explicit cleaning rules; otherwise target labels can remain inconsistent.


# Mark vs Finish Status Disagreement Diagnostic

Audit-only diagnostic for disagreement between `final_mark >= 50` and official `finish_status == "P"`.

This notebook does not write data and does not add mismatch columns as model features. Current target design remains:

- M1 target = `final_mark >= 50`
- M2 target = `final_mark`
- `finish_status` is not used as the main target.

In [ ]:
from pathlib import Path

import pandas as pd

from src.paths import MERGE_DIR

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

# Default audit target: the CRG+ADD+ACD merge output (owned by 01_merge_crg_add_acd).
# If the audit target is an enriched or pre-enriched training file, point DATA_PATH there instead.
DATA_PATH = MERGE_DIR / "merged_add_acd_crg.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")

suffix = DATA_PATH.suffix.lower()
if suffix == ".parquet":
    df = pd.read_parquet(DATA_PATH)
elif suffix == ".csv":
    df = pd.read_csv(DATA_PATH, dtype="string")
else:
    raise ValueError("DATA_PATH must point to a .parquet or .csv file.")

df = df.copy()
df.columns = df.columns.str.lower()

required_columns = {"final_mark", "finish_status"}
missing_required = sorted(required_columns - set(df.columns))
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

final_mark = pd.to_numeric(df["final_mark"], errors="coerce")
finish_status = df["finish_status"].astype("string").str.strip().str.upper().replace("", pd.NA)

analysis_mask = final_mark.notna() & finish_status.notna()
mark_based_pass = final_mark.ge(50)
official_pass = finish_status.eq("P")
mismatch_mask = analysis_mask & mark_based_pass.ne(official_pass)

total_rows = int(analysis_mask.sum())
mismatch_count = int(mismatch_mask.sum())
mismatch_pct = (mismatch_count / total_rows * 100) if total_rows else 0.0

mark_ge_50_not_p = int((analysis_mask & mark_based_pass & ~official_pass).sum())
mark_lt_50_p = int((analysis_mask & ~mark_based_pass & official_pass).sum())

print(f"DATA_PATH: {DATA_PATH}")
print(f"Total rows with non-null final_mark and finish_status: {total_rows:,}")
print(f"Mismatch count: {mismatch_count:,}")
print(f"Mismatch percentage: {mismatch_pct:.4f}%")
print(f"final_mark >= 50 but finish_status != 'P': {mark_ge_50_not_p:,}")
print(f"final_mark < 50 but finish_status == 'P': {mark_lt_50_p:,}")

print("\nMismatch breakdown by finish_status:")
finish_status_breakdown = (
    finish_status[mismatch_mask]
    .value_counts(dropna=False)
    .rename_axis("finish_status")
    .reset_index(name="mismatch_count")
)
finish_status_breakdown["mismatch_pct"] = (
    finish_status_breakdown["mismatch_count"].div(mismatch_count).mul(100) if mismatch_count else 0.0
)
display(finish_status_breakdown)

def mismatch_concentration(column: str, top_n: int = 20) -> pd.DataFrame:
    mismatch_counts = df.loc[mismatch_mask, column].value_counts(dropna=False).rename("mismatch_count")
    total_counts = df.loc[analysis_mask, column].value_counts(dropna=False).rename("total_rows")
    concentration = (
        pd.concat([mismatch_counts, total_counts], axis=1)
        .fillna(0)
        .astype({"mismatch_count": "int64", "total_rows": "int64"})
        .reset_index()
        .rename(columns={"index": column})
    )
    concentration["mismatch_pct_of_all_mismatches"] = (
        concentration["mismatch_count"].div(mismatch_count).mul(100) if mismatch_count else 0.0
    )
    concentration["mismatch_pct_within_group"] = (
        concentration["mismatch_count"].div(concentration["total_rows"].replace(0, pd.NA)).mul(100)
    )
    return concentration.sort_values(
        ["mismatch_count", "mismatch_pct_within_group"],
        ascending=[False, False],
    ).head(top_n)

for column in ["degree_id", "course_id", "part_year", "part_id", "faculty_id"]:
    print(f"\nTop mismatch concentrations by {column}:")
    if column not in df.columns:
        print(f"Column not found: {column}")
        continue
    display(mismatch_concentration(column))